### **Cuaderno A — CLIP como dual encoder: embeddings, retrieval imagen-texto y hard negatives**

**Examen Parcial Virtual MCC225A — Oscar Benito Toledo Guerrero**
**Tema asignado:** CLIP (Radford et al., 2021 — arXiv:2103.00020)

#### Trazabilidad (¿de dónde viene este cuaderno?)

| Sección de este cuaderno | Adaptado de | Qué cambié |
|---|---|---|
| Carga de Flickr8k | **C6**, celda `build_flickr8k_splits` | Tomo **una caption por imagen** para el retrieval alineado (diagonal = positivo) y conservo las 5 para la sección multicaption |
| Carga de OpenCLIP | **C9**, §4 (`create_model`) | Reimplemento `encode_images` / `encode_texts` sin depender de `src/` del proyecto del curso |
| Matriz de similitud y Recall@K | **C9** §6 (`summarize_ranking`) + **C6** (`retrieval_metrics`) | Función propia `recall_at_k` parametrizable, reporta i2t y t2i en una sola tabla |
| Ejemplos de recuperación cruzada | **C9** §7 | Consultas textuales propias |
| Negativos duros | **C9** §8 (`mine_hard_negatives`) | Implementación propia con visualización del par incorrecto |
| Retrieval multicaption | **C10** §3 | Positivos = las 5 captions de la imagen |

**Resultado principal que sustenta este cuaderno:** tabla Recall@{1,5,10} de OpenCLIP `ViT-B-32 / laion2b_s34b_b79k` sobre un subconjunto de Flickr8k, en ambas direcciones, sin ningún entrenamiento (evidencia de la tesis: *el dual-encoder contrastivo es escalable para retrieval y zero-shot*).


#### 0. Requisitos

```bash
pip install open_clip_torch datasets pandas matplotlib
```
Pensado para el contenedor Docker del curso con GPU; funciona también en CPU reduciendo `N_IMAGES`.


In [ ]:
# 0. Setup
import warnings
warnings.filterwarnings("ignore")

import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import open_clip
from datasets import load_dataset

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
N_IMAGES = 200          # tamaño del subconjunto de test (subir/bajar segun hardware)
BATCH_SIZE = 32
MODEL_NAME = "ViT-B-32"
PRETRAINED = "laion2b_s34b_b79k"   # mismo checkpoint baseline que C9

os.makedirs("outputs", exist_ok=True)
os.makedirs("evidencias/metricas", exist_ok=True)
os.makedirs("evidencias/graficos", exist_ok=True)
os.makedirs("evidencias/hard_negatives", exist_ok=True)
os.makedirs("evidencias/rankings", exist_ok=True)
print("DEVICE:", DEVICE)

#### 1. Datos: subconjunto de Flickr8k

Adaptado de `build_flickr8k_splits` (**C6**). Diferencia clave: para evaluar retrieval con pares
alineados 1-a-1 (como el bootstrap de **C9**, donde *la diagonal es el match correcto*), tomo la
`caption_0` de cada imagen. Las 5 captions completas se guardan para la sección 6 (multicaption, como en **C10**).


In [ ]:
# 1. Carga de datos (adaptado de C6: build_flickr8k_splits)
dataset = load_dataset("jxie/flickr8k")     # mismo dataset que C6
test_raw = dataset["test"][:N_IMAGES]

caption_cols = sorted([k for k in test_raw.keys() if k.startswith("caption_")])

rows = []
for i in range(len(test_raw["image"])):
    rows.append({
        "image_id": f"test_{i}",
        "image": test_raw["image"][i],
        "caption": test_raw[caption_cols[0]][i],                      # caption principal (par alineado)
        "all_captions": [test_raw[c][i] for c in caption_cols],      # las 5, para multicaption
    })

meta = pd.DataFrame(rows)
print("imagenes:", len(meta))
meta[["image_id", "caption"]].head()

In [ ]:
# Galeria rapida (adaptado de show_gallery, C9 §3)
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (_, row) in zip(axes, meta.sample(4, random_state=SEED).iterrows()):
    ax.imshow(row["image"])
    ax.set_title(row["caption"][:45] + "...", fontsize=8)
    ax.axis("off")
plt.savefig("evidencias/graficos/galeria_dataset.png", dpi=150, bbox_inches="tight")
plt.show()

#### 2. Cargar OpenCLIP y extraer embeddings

Adaptado de **C9** §4-5. En el proyecto del curso esto vivía en `src/` y `scripts/02_build_embeddings.py`;
aquí lo reimplemento de forma autocontenida. Puntos defendibles:

- `model.encode_image` = torre visual (ViT-B/32); `model.encode_text` = torre textual (transformer con tokenizador BPE, contexto 77).
- **Normalización L2 de ambos lados**: convierte el producto interno en similitud coseno. En el CLIP original
  (`model.py`, `forward`) esto aparece como `logit_scale.exp() * image_features @ text_features.t()`;
  la temperatura no altera el orden del ranking, por eso aquí se omite.


In [ ]:
# 2. Modelo y funciones de codificacion (reemplazan a src/ de C9)
model, _, preprocess = open_clip.create_model_and_transforms(
    MODEL_NAME, pretrained=PRETRAINED, device=DEVICE)
tokenizer = open_clip.get_tokenizer(MODEL_NAME)
model.eval()

@torch.no_grad()
def encode_images(images, batch_size=BATCH_SIZE):
    feats = []
    for i in range(0, len(images), batch_size):
        batch = torch.stack([preprocess(img.convert("RGB")) for img in images[i:i+batch_size]]).to(DEVICE)
        f = model.encode_image(batch)
        f = f / f.norm(dim=-1, keepdim=True)     # normalizacion L2 (clave)
        feats.append(f.cpu())
    return torch.cat(feats).numpy()

@torch.no_grad()
def encode_texts(texts, batch_size=BATCH_SIZE):
    feats = []
    for i in range(0, len(texts), batch_size):
        tokens = tokenizer(texts[i:i+batch_size]).to(DEVICE)
        f = model.encode_text(tokens)
        f = f / f.norm(dim=-1, keepdim=True)
        feats.append(f.cpu())
    return torch.cat(feats).numpy()

image_features = encode_images(meta["image"].tolist())
text_features = encode_texts(meta["caption"].tolist())
print("image_features:", image_features.shape, "| text_features:", text_features.shape)
print("norma media (debe ser ~1.0):", float(np.linalg.norm(image_features, axis=1).mean()))

np.savez("outputs/embeddings_flickr8k.npz",
         image_features=image_features, text_features=text_features,
         model_name=MODEL_NAME, pretrained=PRETRAINED)

#### 3. Matriz de similitud y Recall@K

**La celda clave de mi defensa.** Adaptado de **C9** §6 y de `retrieval_metrics` de **C6**.
Como los pares estan alineados, la diagonal de `sim` es el match correcto.


In [ ]:
# 3. CELDA CLAVE: similitud + Recall@K
sim = image_features @ text_features.T     # (N, N); cos(theta) porque ya estan normalizados

def recall_at_k(sim_matrix, k):
    """Positivo en la diagonal. Adaptacion propia de summarize_ranking (C9) / retrieval_metrics (C6)."""
    ranks = (-sim_matrix).argsort(axis=1)
    hits = [(ranks[i, :k] == i).any() for i in range(sim_matrix.shape[0])]
    return float(np.mean(hits))

KS = (1, 5, 10)
tabla = pd.DataFrame([
    {"direction": "image_to_text", **{f"R@{k}": round(recall_at_k(sim, k), 4) for k in KS}},
    {"direction": "text_to_image", **{f"R@{k}": round(recall_at_k(sim.T, k), 4) for k in KS}},
])
tabla.to_csv("evidencias/metricas/retrieval_recall.csv", index=False)
tabla

In [ ]:
# Posicion media del positivo en el ranking (metrica complementaria)
ranks = (-sim).argsort(axis=1)
pos_rank = np.array([int(np.where(ranks[i] == i)[0][0]) + 1 for i in range(sim.shape[0])])
print(f"rango medio del positivo (i2t): {pos_rank.mean():.2f}  |  mediana: {np.median(pos_rank):.0f}")
plt.figure(figsize=(7, 3))
plt.hist(pos_rank, bins=30)
plt.xlabel("posicion del positivo en el ranking"); plt.ylabel("frecuencia")
plt.title("Distribucion del rango del positivo (image -> text)")
plt.savefig("evidencias/graficos/distribucion_rangos.png", dpi=150, bbox_inches="tight")
plt.show()

#### 4. Ejemplos de recuperación cruzada

Adaptado de **C9** §7. Cambié las consultas por consultas propias. En vivo el docente puede pedir
*"cambia la consulta y explica cómo cambia el ranking"*: solo se recodifica el texto (un `encode_texts`),
las imágenes ya están indexadas — esa es la ventaja operativa del dual encoder.


In [ ]:
# 4a. text -> image
def topk_text_to_image(query, k=5):
    qf = encode_texts([query])
    scores = (qf @ image_features.T)[0]
    idx = np.argsort(-scores)[:k]
    return idx, scores[idx]

query = "a dog running on the grass"      # consulta propia (cambiar en vivo si lo piden)
idx, scores = topk_text_to_image(query, k=5)

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, j, s in zip(axes, idx, scores):
    ax.imshow(meta.iloc[j]["image"]); ax.axis("off")
    ax.set_title(f"score={s:.3f}", fontsize=9)
fig.suptitle(f'Consulta: "{query}"')
plt.savefig("evidencias/graficos/retrieval_t2i.png", dpi=150, bbox_inches="tight")
plt.show()

pd.DataFrame({"rank": range(1, 6), "image_id": meta.iloc[idx]["image_id"].values,
              "score": scores.round(4)}).to_csv("evidencias/rankings/t2i_query1.csv", index=False)

In [ ]:
# 4b. image -> text
def topk_image_to_text(img_idx, k=5):
    scores = sim[img_idx]
    idx = np.argsort(-scores)[:k]
    return pd.DataFrame({
        "rank": range(1, k + 1),
        "caption": meta.iloc[idx]["caption"].values,
        "score": scores[idx].round(4),
        "es_correcta": [j == img_idx for j in idx],
    })

img_idx = 7
plt.figure(figsize=(4, 4)); plt.imshow(meta.iloc[img_idx]["image"]); plt.axis("off")
plt.title("Imagen de consulta"); plt.show()
res = topk_image_to_text(img_idx, k=5)
res.to_csv("evidencias/rankings/i2t_query1.csv", index=False)
res

#### 5. Negativos duros

Adaptado de `mine_hard_negatives` (**C9** §8), implementación propia. Un *hard negative* es un par
incorrecto con score alto: revela **qué confunde al modelo**. La lectura esperada: el negativo comparte
objetos y escena con el positivo, pero difiere en composición o detalle — limitación estructural del
dual encoder (sin interacción cruzada entre tokens).


In [ ]:
# 5. Mineria de hard negatives (implementacion propia)
def mine_hard_negatives(sim_matrix, top_n=8):
    rows = []
    for i in range(sim_matrix.shape[0]):
        scores = sim_matrix[i].copy()
        scores[i] = -np.inf                      # excluir el positivo
        j = int(np.argmax(scores))               # negativo con mayor score
        rows.append({
            "image_id": meta.iloc[i]["image_id"],
            "caption_correcta": meta.iloc[i]["caption"],
            "negative_caption": meta.iloc[j]["caption"],
            "score_negativo": float(sim_matrix[i, j]),
            "score_positivo": float(sim_matrix[i, i]),
            "positivo_gana": bool(sim_matrix[i, i] > sim_matrix[i, j]),
            "neg_idx": j, "img_idx": i,
        })
    df = pd.DataFrame(rows).sort_values("score_negativo", ascending=False).head(top_n)
    return df.reset_index(drop=True)

hard = mine_hard_negatives(sim, top_n=8)
hard.to_csv("evidencias/hard_negatives/hard_negatives.csv", index=False)
hard[["image_id", "caption_correcta", "negative_caption", "score_negativo", "positivo_gana"]]

In [ ]:
# Visualizar el hard negative principal
row = hard.iloc[0]
plt.figure(figsize=(5, 5))
plt.imshow(meta.iloc[int(row["img_idx"])]["image"]); plt.axis("off")
plt.title(f'CORRECTA: {row["caption_correcta"][:60]}\nNEGATIVO (score {row["score_negativo"]:.3f}): {row["negative_caption"][:60]}',
          fontsize=9)
plt.savefig("evidencias/hard_negatives/hard_negative_top1.png", dpi=150, bbox_inches="tight")
plt.show()

#### 6. Retrieval multicaption (estructura correcta del problema)

Adaptado de **C10** §3: una imagen tiene 5 captions válidas, así que en image→text el acierto es
recuperar **cualquiera** de sus 5 positivos. Esto suele subir Recall@K respecto a la evaluación 1-a-1
y es la forma honesta de medir en Flickr.


In [ ]:
# 6. Multicaption: explotar las 5 captions y recalcular metricas
all_caps, cap_to_img = [], []
for i, caps in enumerate(meta["all_captions"]):
    for c in caps:
        all_caps.append(c)
        cap_to_img.append(i)
cap_to_img = np.array(cap_to_img)

text_features_mc = encode_texts(all_caps)
sim_mc = image_features @ text_features_mc.T      # (N_img, N_caps)

def recall_at_k_multicaption(sim_matrix, positives_per_query, k):
    ranks = (-sim_matrix).argsort(axis=1)
    hits = []
    for q in range(sim_matrix.shape[0]):
        topk = set(ranks[q, :k].tolist())
        hits.append(len(topk & set(positives_per_query[q])) > 0)
    return float(np.mean(hits))

img_pos = {i: np.where(cap_to_img == i)[0].tolist() for i in range(len(meta))}        # i2t: 5 positivos
txt_pos = {t: [int(cap_to_img[t])] for t in range(len(all_caps))}                     # t2i: 1 positivo

tabla_mc = pd.DataFrame([
    {"direction": "image_to_text (multicaption)",
     **{f"R@{k}": round(recall_at_k_multicaption(sim_mc, img_pos, k), 4) for k in KS}},
    {"direction": "text_to_image (multicaption)",
     **{f"R@{k}": round(recall_at_k_multicaption(sim_mc.T, txt_pos, k), 4) for k in KS}},
])
tabla_mc.to_csv("evidencias/metricas/retrieval_recall_multicaption.csv", index=False)
tabla_mc

#### 7. Cierre: relación con el paper, limitación y mejora

**Relación con CLIP (arXiv:2103.00020):**
- `encode_image` / `encode_text` = las dos torres de la Fig. 1; normalización L2 + producto interno = la similitud coseno de la pérdida InfoNCE (aquí solo en inferencia: el preentrenamiento con 400M de pares WIT no se reproduce, se *usa*).
- El retrieval sin entrenamiento sobre Flickr8k evidencia la **transferencia** que defiende mi tesis.

**Limitación encontrada:** subconjunto pequeño (varianza alta en R@K) y hard negatives que muestran fallas de composición fina — el dual encoder empareja objetos/escena, no relaciones.

**Mejora propuesta:** (1) subir `N_IMAGES` y repetir con varios seeds; (2) re-ranking del top-k con un cross-encoder (ver Cuaderno C); (3) consultas/prompts más específicos.
